# Lab 10 — Build a Simple Engineering Agent
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 10 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab10-engineering-agent/Lab10_Engineering_Agent.ipynb)

Welcome back from spring break! A chatbot **answers**; an **agent** **acts**. It picks a tool, runs it, reads the result, and decides what to do next, again and again, until the task is done. Today you build one from scratch in plain Python: a few engineering tools, a loop, and a "brain" that decides which tool to call. First the brain is a set of `if` rules, so you can see every step. Then the brain is a small language model. You will test the agent on five tasks, keep score honestly, and add the two guardrails every real agent needs: a step limit and human approval for actions that matter.

**Final project:** the project is assigned this week. Teams in the **Assist** track can start from this notebook.

### What you will be able to do
1. Explain the agent loop (observe → think → act → observe) and follow it step by step in a trace.
2. Write engineering tools as Python functions with docstrings that state inputs, outputs, and units.
3. Connect a small LLM as the brain that requests tools in JSON, and find its mistakes in the trace.
4. Evaluate an agent on test tasks, and add guardrails: a step limit and human approval for consequential actions.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `concrete_strength.csv` (1,030 lab-tested concrete mixes, CC BY 4.0) and `osha/osha_1926_subpartP_excavations.txt` (OSHA excavation rules, U.S. Government work).

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import ast, gc, inspect, json, re, time
import numpy as np
import pandas as pd
import torch
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — Tools are just Python functions (15 min)
A **tool** is a function the agent is allowed to call. The model never runs anything itself. It only *asks*, and
**your code** runs the function. Good tools have:
- a clear **name**,
- **arguments with units in their names** (`L_m`, `w_kN_per_m`), so nobody guesses,
- a **docstring** (the text in triple quotes). The docstring is all the model will know about the tool, so it must
  say what goes in, what comes out, and in which units.

### Tool 1: a unit converter

In [ ]:
# Each unit -> (kind of quantity, how many SI base units it equals)
UNITS = {
    "m": ("length", 1.0), "mm": ("length", 0.001), "cm": ("length", 0.01), "km": ("length", 1000.0),
    "ft": ("length", 0.3048), "in": ("length", 0.0254), "yd": ("length", 0.9144),
    "kN": ("force", 1.0), "N": ("force", 0.001), "kip": ("force", 4.448222), "lbf": ("force", 0.004448222),
    "MPa": ("stress", 1000.0), "kPa": ("stress", 1.0), "psi": ("stress", 6.894757), "ksi": ("stress", 6894.757),
    "psf": ("stress", 0.04788026),
    "kN/m": ("line load", 1.0), "kip/ft": ("line load", 14.59390), "lb/ft": ("line load", 0.01459390),
    "m3": ("volume", 1.0), "yd3": ("volume", 0.7645549), "ft3": ("volume", 0.02831685), "L": ("volume", 0.001),
}

def convert_units(value, from_unit, to_unit):
    """Convert a number between two units of the same kind of quantity.
    Units: length m, mm, cm, km, ft, in, yd; force kN, N, kip, lbf; stress MPa, kPa, psi, ksi, psf;
    line load kN/m, kip/ft, lb/ft; volume m3, yd3, ft3, L.
    Example: convert_units(12, "ft", "m") returns 3.6576"""
    if from_unit not in UNITS or to_unit not in UNITS:
        raise ValueError(f"unknown unit. Use one of: {', '.join(UNITS)}")
    kind_from, factor_from = UNITS[from_unit]
    kind_to, factor_to = UNITS[to_unit]
    if kind_from != kind_to:
        raise ValueError(f"cannot convert {kind_from} ({from_unit}) to {kind_to} ({to_unit})")
    return round(float(value) * factor_from / factor_to, 4)

print(convert_units(12, "ft", "m"), "m")
print(convert_units(4000, "psi", "MPa"), "MPa")

### Tool 2: a simply supported beam with a uniform load
For a simply supported beam with a uniform load *w* over span *L*, both maxima occur at midspan:

- maximum bending moment: **M = w L² / 8**
- maximum deflection: **δ = 5 w L⁴ / (384 E I)**

where *E* is the modulus of elasticity and *I* is the second moment of area. The formula only works if the units
match, so the function converts everything to kN and m first: E in GPa × 10⁶ = kN/m², and I in mm⁴ × 10⁻¹² = m⁴.

> 💡 **Hint:** `**` means power: `L**2` is L². Use the variables `w`, `L`, `E`, `I` that are already defined inside the function.

In [ ]:
# TODO: complete the two formulas (replace the None values).
def beam_max_moment_and_deflection(w_kN_per_m, L_m, E_GPa=None, I_mm4=None):
    """Simply supported beam with a uniform load w (kN/m) over a span L (m).
    Returns the maximum bending moment M_max_kNm = w*L^2/8 (kN·m, at midspan). If E_GPa (GPa) and
    I_mm4 (mm^4) are also given, also returns the maximum deflection deflection_mm = 5*w*L^4/(384*E*I) (mm)."""
    w, L = float(w_kN_per_m), float(L_m)
    M = None                                    # kN·m
    result = {"M_max_kNm": round(M, 2)}
    if E_GPa is not None and I_mm4 is not None:
        E = float(E_GPa) * 1e6                  # GPa  -> kN/m²
        I = float(I_mm4) * 1e-12                # mm⁴  -> m⁴
        delta_m = None                          # m
        result["deflection_mm"] = round(delta_m * 1000, 2)
    return result

beam_max_moment_and_deflection(12, 6, E_GPa=200, I_mm4=80e6)

In [ ]:
# ✅ Self-check: run this cell to test your work above
r = beam_max_moment_and_deflection(12, 6, E_GPa=200, I_mm4=80e6)
assert abs(r["M_max_kNm"] - 54.0) < 0.01, "M should be 12 × 6² / 8 = 54 kN·m"
assert abs(r["deflection_mm"] - 12.66) < 0.02, "deflection should be about 12.66 mm — check the 5, 384 and L**4"
print("Correct! Deflection limit L/360 =", round(6000 / 360, 1), "mm, so this beam passes.")

### Tool 3: an OSHA lookup (the Lab 08 retriever)
This is the chunker and TF-IDF search from Lab 08, packed into one tool. It returns the two best-matching
passages as plain, numbered text (like the Lab 08 RAG prompt), because the model will have to read it. **The tool
only finds text; it does not decide what the text means.**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def make_chunks(text, min_words=50):
    """Lab 08 chunker: whole paragraphs, >= min_words per chunk, table rows kept with their title."""
    paragraphs, section = [], "Subpart P"
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("## "):
            section = line[3:]
        elif " | " in line and paragraphs and paragraphs[-1][0] == section:
            paragraphs[-1][1] += "\n" + line
        else:
            paragraphs.append([section, line])
    chunks, current, current_section = [], [], None
    for section, para in paragraphs:
        if current and (section != current_section or len(" ".join(current).split()) >= min_words):
            chunks.append({"section": current_section, "text": "\n".join(current)})
            current = []
        current_section = section
        current.append(para)
    chunks.append({"section": current_section, "text": "\n".join(current)})
    return chunks

osha_chunks = make_chunks(open(get_data("osha/osha_1926_subpartP_excavations.txt"), encoding="utf-8").read())
osha_vectorizer = TfidfVectorizer(stop_words="english")
osha_vectors = osha_vectorizer.fit_transform([c["text"] for c in osha_chunks])

def search_osha(query):
    """Keyword (TF-IDF) search in OSHA 29 CFR 1926 Subpart P (Excavations). Returns the 2 best-matching
    passages as numbered text, each with its section name. Works best with the rule's own words, for example
    "maximum allowable slopes Type C soil excavations less than 20 feet deep"."""
    scores = cosine_similarity(osha_vectorizer.transform([str(query)]), osha_vectors)[0]
    best = np.argsort(scores)[::-1][:2]
    return "\n\n".join(f"[{r + 1}] ({osha_chunks[i]['section']})\n{osha_chunks[i]['text']}"
                       for r, i in enumerate(best))

print(search_osha("maximum allowable slopes Type C soil excavations less than 20 feet deep"))

### Tool 4: a concrete-strength predictor (a machine-learning model as a tool)
Agents can call ML models too. We train a **random forest** (many decision trees that vote) on the concrete data
from Labs 02–05, test it on mixes it has not seen, and wrap it in a function.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

concrete = pd.read_csv(get_data("concrete_strength.csv"))
FEATURES = ["cement_kg_m3", "slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3", "superplasticizer_kg_m3",
            "coarse_agg_kg_m3", "fine_agg_kg_m3", "age_days"]
X_train, X_test, y_train, y_test = train_test_split(concrete[FEATURES], concrete["strength_mpa"],
                                                    test_size=0.2, random_state=42)
forest = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
mae = mean_absolute_error(y_test, forest.predict(X_test))
print(f"Test MAE = {mae:.1f} MPa on {len(y_test)} mixes the model never saw")
TYPICAL = concrete[FEATURES].median()      # typical aggregate amounts, used when the user gives none

def predict_concrete_strength(cement_kg_m3, water_kg_m3, age_days, slag_kg_m3=0, fly_ash_kg_m3=0,
                              superplasticizer_kg_m3=0):
    """Predict the compressive strength (MPa) of a concrete mix with a random-forest model trained on 1,030
    lab tests. Inputs: amounts in kg per m^3 of concrete, and the age at testing in days (1 to 365).
    Aggregates are set to typical amounts. Typical error is about +/- 4 MPa. Only valid for mixes similar
    to the training data."""
    mix = {"cement_kg_m3": cement_kg_m3, "slag_kg_m3": slag_kg_m3, "fly_ash_kg_m3": fly_ash_kg_m3,
           "water_kg_m3": water_kg_m3, "superplasticizer_kg_m3": superplasticizer_kg_m3,
           "coarse_agg_kg_m3": TYPICAL["coarse_agg_kg_m3"], "fine_agg_kg_m3": TYPICAL["fine_agg_kg_m3"],
           "age_days": age_days}
    x = pd.DataFrame([mix])[FEATURES].astype(float)
    return {"strength_MPa": round(float(forest.predict(x)[0]), 1)}

predict_concrete_strength(cement_kg_m3=350, water_kg_m3=175, age_days=28)

### Tool 5: a door-schedule query (BIM)
**BIM (building information modeling)** software such as Revit stores a building as a database, and a **door
schedule** is a table of every door: its mark (ID), level, room, size, and **fire rating** (how many minutes the
door assembly resists fire in a standard test). Questions like "how many 90-minute doors are on Level 2?" are
classic agent tasks. The schedule below is **synthetic**: we made it up for this lab, in the shape of a Revit
export. It is not a real building, and it is not a code check.

In [ ]:
# SYNTHETIC door schedule (made up for this lab): 3 levels x 10 doors
rooms = ["Stair 1", "Stair 2", "Elevator lobby", "Corridor", "Mechanical", "Electrical",
         "Office", "Office", "Restroom", "Storage"]
widths_mm = [915, 915, 1830, 1830, 915, 915, 915, 915, 915, 915]
ratings_min = {1: [90, 90, 90, 20, 60, 60, 0, 0, 0, 45],     # fire rating of each door, minutes
               2: [90, 90, 90, 20, 90, 60, 0, 0, 0, 0],
               3: [90, 90, 60, 20, 60, 45, 0, 0, 0, 0]}
doors = pd.DataFrame([{"mark": f"D{level}{i + 1:02d}", "level": level, "room": rooms[i],
                       "width_mm": widths_mm[i], "fire_rating_min": ratings_min[level][i]}
                      for level in (1, 2, 3) for i in range(10)])

def count_doors(level, min_fire_rating_min=0):
    """Count the doors on one level of the project's door schedule (a synthetic Revit-style schedule) whose fire
    rating is at least min_fire_rating_min minutes (0 = all doors). level: 1, 2, or 3. Returns the count and
    the door marks."""
    level = int(re.findall(r"\d+", str(level))[0])       # accepts 2, "2", "Level 2", "L2"
    hits = doors[(doors["level"] == level) & (doors["fire_rating_min"] >= float(min_fire_rating_min))]
    return {"level": level, "count": int(len(hits)), "marks": list(hits["mark"])}

print(doors.head(10).to_string(index=False))
count_doors("Level 1", 60)

> 💡 **Hint:** `(doors["level"] == 2) & (doors["fire_rating_min"] >= 90)` is True for each matching row; `.sum()` counts the Trues.

In [ ]:
# TODO: answer "How many doors on Level 2 have a fire rating of at least 90 minutes?" with pandas alone
# (no tool). Combine two conditions with &, then count the rows with .sum().
n_level2_90 = None
print(n_level2_90)

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert n_level2_90 == count_doors(2, 90)["count"], "Your pandas count and the tool disagree. Check both conditions."
print("Correct! This is the answer we will expect from the agent in Part 4.")

### The toolbox
The agent gets a dictionary `TOOLS` that maps each tool's name to its function. `describe_tools()` turns the
names, arguments, and docstrings into the text the model will read in Part 3.

In [ ]:
TOOLS = {f.__name__: f for f in [convert_units, beam_max_moment_and_deflection,
                                  search_osha, predict_concrete_strength, count_doors]}

def describe_tools(tools):
    """One line per tool: name(arguments): docstring."""
    return "\n".join(f"- {name}{inspect.signature(f)}: {' '.join(inspect.getdoc(f).split())}"
                     for name, f in tools.items())

print(describe_tools(TOOLS))

**✍️ Question:** Why do the argument names include units (`L_m`, `E_GPa`, `I_mm4`)? What could go wrong if the beam tool just had arguments `L`, `E`, and `I`?

*Your answer:* 

## Part 2 — The agent loop, with a rule-based brain (10 min)
Every agent runs the same loop:

1. **Observe**: read the task and everything that has happened so far (the *history*).
2. **Think**: the *brain* decides the next action, either "call this tool with these arguments" or "here is the
   final answer".
3. **Act**: *our code* runs the tool (only tools in `TOOLS`), and the result becomes a new **observation**.
4. Repeat, until there is a final answer or we hit the **step limit**.

The brain answers in **JSON**, a text format for data that looks like a Python dictionary:
`{"tool": "convert_units", "args": {"value": 12, "from_unit": "ft", "to_unit": "m"}}` or
`{"final_answer": "12 ft = 3.66 m"}`. If the brain replies with plain text and no JSON at all, the loop treats that
text as the final answer (the usual convention: "no tool call = done"). The cell below is the whole loop, about 40
lines. Read the comments.

In [ ]:
CONSEQUENTIAL = set()      # names of tools that need a human's OK before they run (Part 5)

def human_approval(tool_name, args):
    """Ask a human to approve a consequential action. If no human answers, the answer is NO."""
    try:
        reply = input(f"    ⚠️  Agent wants to run {tool_name}({args}). Type yes to allow: ")
    except Exception:          # no keyboard available (e.g., automatic test run)
        reply = ""
        print("    (no human available, so the safe default is NO)")
    return reply.strip().lower() == "yes"

def run_tool(name, args):
    """Run one tool safely and return the observation as text (errors become observations too)."""
    if name not in TOOLS:
        return f"ERROR: there is no tool named {name!r}. Available tools: {', '.join(TOOLS)}"
    if not isinstance(args, dict):
        return "ERROR: args must be a JSON object like {\"name\": value}"
    if name in CONSEQUENTIAL and not human_approval(name, args):
        return "DENIED: a human did not approve this action. Do not retry it; report this in the final answer."
    try:
        result = TOOLS[name](**args)
        return result if isinstance(result, str) else json.dumps(result, ensure_ascii=False)   # text stays text
    except Exception as err:
        return f"ERROR: {type(err).__name__}: {err}"

def run_agent(task, brain, max_steps=5, verbose=True):
    """Run the observe -> think -> act loop. Returns a summary dictionary."""
    history, tools_used = [], []
    if verbose:
        print(f"📋 TASK: {task}")
    for step in range(1, max_steps + 1):
        reply = brain(task, history)                        # THINK: the brain writes its decision as text
        decision = parse_reply(reply)                       # read the JSON (lenient, see Part 3)
        if verbose:
            print(f"\n— step {step} —\n🧠 brain says: {reply.strip()[:300]}")
        if decision is None and "{" not in reply:              # plain text, no JSON at all:
            decision = {"final_answer": reply.strip()}          # the usual rule is "no tool call = final answer"
        if decision is None:
            observation = "ERROR: I could not read your reply as JSON. Reply with ONE JSON object."
        elif "final_answer" in decision:
            if verbose:
                print(f"✅ FINAL ANSWER: {decision['final_answer']}")
            return {"final_answer": str(decision["final_answer"]), "steps": step, "tools_used": tools_used}
        else:
            name, args = decision.get("tool"), decision.get("args", {})
            tools_used.append(str(name))
            observation = run_tool(name, args)                # ACT: our code runs the tool
        if verbose:
            print(f"👀 observation: {observation[:300]}")
        history.append({"reply": reply, "observation": observation})   # OBSERVE: remember what happened
    if verbose:
        print(f"\n⛔ STOPPED: reached the limit of {max_steps} steps without a final answer.")
    return {"final_answer": None, "steps": max_steps, "tools_used": tools_used}

`parse_reply` reads the brain's text as JSON. Real models often wrap JSON in extra words or code fences,
use single quotes, or forget a closing brace, so this version is **lenient**: it finds the first `{...}` block and
tries a few repairs before giving up.

In [ ]:
def parse_reply(text):
    """Find the first {...} block in the text and read it as a dictionary. Returns None if impossible."""
    text = re.sub(r"```(json)?", "", str(text)).strip()    # remove code fences
    try:                                                  # easy case: the whole reply is clean JSON
        result = json.loads(text)
        return result if isinstance(result, dict) else None
    except Exception:
        pass
    start = text.find("{")
    if start == -1:
        return None
    depth, end = 0, None
    for i in range(start, len(text)):                     # find the matching closing brace
        depth += {"{": 1, "}": -1}.get(text[i], 0)
        if depth == 0:
            end = i + 1
            break
    block = text[start:end] if end else text[start:] + "}" * depth    # add missing braces
    block = re.sub(r",\s*}", "}", block)                   # remove trailing commas
    for attempt in (block, block.replace("true", "True").replace("false", "False").replace("null", "None")):
        try:
            return json.loads(attempt)
        except Exception:
            pass
        try:
            result = ast.literal_eval(attempt)             # accepts single quotes, like Python
            return result if isinstance(result, dict) else None
        except Exception:
            pass
    return None

tests = ['{"final_answer": "3.66 m"}',
         'Sure! Here is the call: ```json\n{"tool": "convert_units", "args": {"value": 12, "from_unit": "ft", "to_unit": "m"}}\n```',
         "{'tool': 'search_osha', 'args': {'query': 'Type C slope'}}",
         '{"tool": "convert_units", "args": {"value": 12, "from_unit": "ft", "to_unit": "m"}',
         "I think the answer is 54 kN·m."]
for t in tests:
    print(repr(parse_reply(t)))

Now a **rule-based brain**: plain `if` statements that look for keywords and pull numbers out of the
task with regular expressions (`re.findall`). It has no intelligence at all. It is useful because it makes the
loop's mechanics visible and predictable.

In [ ]:
def numbers_in(text):
    """All numbers in a string, e.g. 'w = 12 kN/m, L = 6 m' -> [12.0, 6.0]."""
    return [float(x.replace(",", "")) for x in re.findall(r"\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text)]

def rule_brain(task, history):
    """A brain made of if-statements. Returns a JSON decision as text."""
    t = task.lower()
    if history:                                   # a tool has already run: report its result and stop
        return json.dumps({"final_answer": f"Tool result: {history[-1]['observation'][:400]}"})
    m = re.search(r"convert ([\d.,]+) ?([a-z0-9/]+) to ([a-z0-9/]+)", task, flags=re.IGNORECASE)
    if m:
        value, unit_from, unit_to = m.groups()
        return json.dumps({"tool": "convert_units",
                           "args": {"value": float(value.replace(",", "")), "from_unit": unit_from, "to_unit": unit_to}})
    if "beam" in t and len(numbers_in(task)) >= 2:
        n = numbers_in(task)
        args = {"w_kN_per_m": n[1], "L_m": n[0]}    # assumes the span comes first, then the load!
        if len(n) >= 4:
            args.update({"E_GPa": n[2], "I_mm4": n[3]})
        return json.dumps({"tool": "beam_max_moment_and_deflection", "args": args})
    if "door" in t:
        level = re.search(r"level\s*(\d+)", t)
        rating = re.search(r"(\d+)[- ]?min", t)
        return json.dumps({"tool": "count_doors",
                           "args": {"level": int(level.group(1)) if level else 1,
                                    "min_fire_rating_min": int(rating.group(1)) if rating else 0}})
    if "osha" in t or "excavation" in t or "trench" in t:
        return json.dumps({"tool": "search_osha", "args": {"query": task}})
    if "strength" in t and len(numbers_in(task)) >= 3:
        n = numbers_in(task)
        return json.dumps({"tool": "predict_concrete_strength",
                           "args": {"age_days": n[0], "cement_kg_m3": n[1], "water_kg_m3": n[2]}})
    return json.dumps({"final_answer": "Sorry, I have no rule for this task."})

run_agent("Convert 12 ft to m", rule_brain)

In [ ]:
run_agent("A simply supported beam spans 6 m and carries 12 kN/m. E = 200 GPa and I = 80,000,000 mm^4. "
          "Find the maximum moment and deflection.", rule_brain)

> 💡 **Hint:** Try a different wording for a unit conversion ("How many meters are in 12 feet?"), or a beam task that gives the load before the span.

In [ ]:
# TODO: write a task that a human would understand but that BREAKS the rule brain
# (wrong tool, wrong numbers, or "no rule"). Run it and look at the trace.
my_task = "..."
run_agent(my_task, rule_brain)

**✍️ Question:** Why did the rule brain fail on your task? What is the main strength and the main weakness of a rule-based brain?

*Your answer:* 

## Part 3 — An LLM brain (12 min)
Now we swap the `if` statements for a language model. The model reads a **system prompt** that lists the tools
(the docstrings from Part 1) and the JSON format, then the task, then every earlier reply and observation. On a
GPU runtime this uses the bigger 1.5B model. On CPU it uses the 0.5B model from Labs 01 and 08 (about 1 GB, 1–2
minutes to load). Model downloads happen on Google's computers, not on your laptop or your phone's data plan.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if torch.cuda.is_available() else "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
llm = AutoModelForCausalLM.from_pretrained(MODEL).to("cuda" if torch.cuda.is_available() else "cpu")
llm.eval()
print("Loaded", MODEL)

def make_system_prompt(tools):
    """The instructions the LLM reads before every decision: the tool list and the JSON reply format."""
    return f"""You are an engineering assistant that solves tasks by calling tools.
Available tools:
{describe_tools(tools)}

Reply with exactly ONE JSON object and nothing else.
To call a tool: {{"tool": "tool_name", "args": {{"argument_name": value}}}}
When you know the answer: {{"final_answer": "the answer, with units"}}
Use the numbers that the tools return. Do not do the math in your head."""

SYSTEM_PROMPT = make_system_prompt(TOOLS)
print(SYSTEM_PROMPT)

**Watch for this:** the system prompt asks for JSON, but small models often write their *final* answer as
a plain sentence. Our loop counts a reply with no JSON at all as the final answer (a common convention), so a
plain-text answer after a tool call is fine. JSON replies are short, so we let the model write at most 80 new
tokens per step, which keeps each step fast.

In [ ]:
def llm_brain(task, history, max_new_tokens=80):
    """Ask the LLM for the next decision, given the task and everything that happened so far."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "Task: " + task}]
    for h in history:
        messages.append({"role": "assistant", "content": h["reply"]})
        messages.append({"role": "user", "content": "Observation: " + h["observation"]})
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                     return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

run_agent("Convert 12 ft to m", llm_brain)

In [ ]:
run_agent("A simply supported beam spans 6 m and carries 12 kN/m. E = 200 GPa and I = 80,000,000 mm^4. "
          "Find the maximum moment and deflection.", llm_brain)

**✍️ Question:** Read the two traces step by step. For each step, did the LLM pick the right tool and the right arguments (with the right units)? Did the final answer use the tool's numbers?

*Your answer:* 

Now the agent version of Lab 08's RAG: the LLM must call `search_osha`, read the passages it gets back,
and answer from them. The task names the tool and gives search words that match a plain-sentence rule in
Appendix B (Figure B-1.3, "Excavations Made in Type C Soil"). Both help the small model. (When we asked it to read
Table B-1 instead, it kept picking the Type A row.)

**Tool names matter.** While we built this lab, the tool was first called `osha_lookup`. The 0.5B model could not
spell that name: even when the task said "osha_lookup", it called "oshawalk", "oshawalkup", or "oshaworkspace".
Renamed `search_osha`, it calls the tool correctly. Names made of plain, common words are part of good tool design.

In [ ]:
run_agent('Use search_osha with the query "Excavations Made in Type C Soil simple slope excavations 20 '
          'feet or less in depth" to find the maximum allowable slope for a simple slope excavation in Type C '
          'soil 20 feet or less deep.', llm_brain)

**✍️ Question:** Did the agent call `search_osha` with the right arguments? Which passage holds the answer, and does the final answer match it (1½H:1V)? What else was in the observation that could have misled it?

*Your answer:* 

## Part 4 — Test the agent on 6 tasks (15 min)
One good-looking trace proves nothing. Engineers test tools on **cases with known answers**. Each test task below
has the correct numbers, worked out by hand or with the tools directly. `check_answer` passes a final answer if
every expected number appears in it (within 1%).

In [ ]:
TEST_TASKS = [
    {"task": "Convert 30 ft to meters.",
     "expected": [9.144], "correct": "9.144 m"},
    {"task": "A simply supported steel beam spans 8 m and carries a uniform load of 10 kN/m. "
             "E = 200 GPa and I = 120,000,000 mm^4. Find the maximum bending moment and the maximum deflection.",
     "expected": [80.0, 22.22], "correct": "M = 80 kN·m, deflection = 22.2 mm"},
    {"task": "According to OSHA, what is the maximum allowable slope for an excavation in Type C soil "
             "less than 20 feet deep?",
     "expected": [1.5], "correct": "1½H:1V (34°), Table B-1"},
    {"task": "Predict the 28-day compressive strength of a concrete mix with 350 kg/m3 of cement and "
             "175 kg/m3 of water, no other binders.",
     "expected": [predict_concrete_strength(350, 175, 28)["strength_MPa"]], "correct": None},
    {"task": "A beam spans 20 ft and carries a uniform load of 15 kN/m. Convert the span to meters, "
             "then find the maximum bending moment.",
     "expected": [69.68], "correct": "L = 6.096 m, M = 69.7 kN·m"},
    {"task": "How many 90-minute fire-rated doors are on Level 2?",
     "expected": [count_doors(2, 90)["count"]], "correct": None},
]
TEST_TASKS[5]["correct"] = f"{count_doors(2, 90)['count']} doors ({', '.join(count_doors(2, 90)['marks'])})"
TEST_TASKS[3]["correct"] = f"{TEST_TASKS[3]['expected'][0]} MPa (the tool's prediction)"

def check_answer(final_answer, expected, rel_tol=0.01):
    """True if every expected number appears in the final answer (within rel_tol). Fractions like 1 1/2 count."""
    if final_answer is None:
        return False
    text = re.sub(r"(\d+)\s*(?:1/2|½)", lambda m: str(int(m.group(1)) + 0.5), str(final_answer))
    text = text.replace("½", "0.5")
    found = numbers_in(text)
    return all(any(abs(f - e) <= rel_tol * abs(e) for f in found) for e in expected)

pd.DataFrame(TEST_TASKS)[["task", "correct"]]

The next cell runs all 6 tasks with both brains and prints the LLM traces, so you can find out *why* a
task failed. **On a CPU it takes several minutes. Start it now, then scroll up and answer the ✍️ questions in
Parts 2–3 while it runs.** Come back when the `[*]` next to the cell turns into a number.

In [ ]:
def evaluate(brain, name, max_steps=5, verbose=False):
    rows = []
    for t in TEST_TASKS:
        t0 = time.time()
        if verbose:
            print("=" * 100)
        r = run_agent(t["task"], brain, max_steps=max_steps, verbose=verbose)
        rows.append({"brain": name, "task": t["task"][:45] + "…", "correct answer": t["correct"],
                     "agent's final answer": (r["final_answer"] or "(none — step limit)")[:120],
                     "tools used": ", ".join(r["tools_used"]) or "(none)", "steps": r["steps"],
                     "auto-check": "✅ pass" if check_answer(r["final_answer"], t["expected"]) else "❌ fail",
                     "seconds": round(time.time() - t0)})
    return pd.DataFrame(rows)

rule_results = evaluate(rule_brain, "rules")
llm_results = evaluate(llm_brain, "LLM", verbose=True)

In [ ]:
results = pd.concat([rule_results, llm_results], ignore_index=True)
pd.set_option("display.max_colwidth", 120)
results

In [ ]:
summary = results.groupby("brain")["auto-check"].apply(lambda s: f"{(s == '✅ pass').sum()} / {len(s)}")
print("Tasks passed:")
print(summary.to_string())

**Do not trust the auto-check blindly either.** It only looks for numbers. It can pass an answer that
contains the right number by luck, or one where the number is buried in a wrong sentence. It can also fail a
correct answer written in a different unit. Read every final answer yourself.

> 💡 **Hint:** Compare each final answer with the "correct answer" column. "Partly" = right number but wrong units or explanation, or only half of a two-part answer.

In [ ]:
# TODO: read the LLM rows above and give YOUR verdict for each task: "correct", "partly", or "wrong".
# Then compare with the auto-check.
my_verdicts = ["?", "?", "?", "?", "?", "?"]      # one per test task, in order
llm_results["my verdict"] = my_verdicts
llm_results[["task", "agent's final answer", "auto-check", "my verdict"]]

**✍️ Question:** Which kinds of tasks did the LLM agent handle, and where did it fail? Pick one failure and say exactly which step went wrong (wrong tool, wrong argument, ignored the observation, made up a number, or never finished).

*Your answer:* 

**✍️ Question:** If you rerun the LLM agent tomorrow with the same settings, would the success table be the same? How many test tasks would you want before trusting an agent on real project work?

*Your answer:* 

## Part 5 — Guardrails: a step limit and human approval (8 min)
**Step limit.** An agent can get stuck, calling the same tool again and again, and each step costs time and money.
`run_agent` already stops after `max_steps`. Here is a deliberately stubborn brain:

In [ ]:
def stubborn_brain(task, history):
    """A broken brain that never gives a final answer."""
    return json.dumps({"tool": "convert_units", "args": {"value": 1, "from_unit": "ft", "to_unit": "m"}})

run_agent("Convert 1 ft to m", stubborn_brain, max_steps=3)

**Human approval.** Some tools *do* things in the world: they send an email, change a model, or order
materials. Those are **consequential**: hard or impossible to undo, and they cost money or affect safety. Here is a
(pretend) ordering tool. Real agents should never run tools like this without a person saying yes.

In [ ]:
def place_concrete_order(volume_m3, strength_MPa, delivery_date):
    """Place a ready-mix concrete order with the supplier: volume (m^3), specified strength (MPa), delivery date.
    This commits real money and cannot be undone."""
    return {"status": "ORDER PLACED (pretend)", "volume_m3": volume_m3, "strength_MPa": strength_MPa,
            "delivery_date": delivery_date}

TOOLS["place_concrete_order"] = place_concrete_order
SYSTEM_PROMPT = make_system_prompt(TOOLS)       # the LLM now knows about the new tool
print(run_tool("place_concrete_order", {"volume_m3": 8, "strength_MPa": 30, "delivery_date": "next Tuesday"}))

Without a guardrail, the order went straight through. Turn the guardrail on.

> 💡 **Hint:** A set with one string looks like `{"place_concrete_order"}`.

In [ ]:
# TODO: make place_concrete_order a consequential tool, so run_tool asks a human first.
# In Colab a box appears when you run this cell: type no (later, try yes and see what changes).
CONSEQUENTIAL = set()     # put the tool's name (a string) inside the braces

print(run_tool("place_concrete_order", {"volume_m3": 8, "strength_MPa": 30, "delivery_date": "next Tuesday"}))

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert "place_concrete_order" in CONSEQUENTIAL, "Add the tool name to the CONSEQUENTIAL set."
print("Guardrail on: the agent must ask before ordering.")

Now give the whole agent an ordering task. (The task names the tool, so that the small model actually tries it.) In Colab, type `no` when the box appears. Watch the trace: which arguments does the LLM send, and what does it do after the observation says DENIED?

In [ ]:
order_task = "Use place_concrete_order to order 8 m3 of 30 MPa concrete for delivery next Tuesday."
run_agent(order_task, llm_brain, max_steps=3)

**✍️ Question:** Look at the arguments in the agent's request. Is anything missing or wrong? What would have happened without the approval step? What did the agent say after it was denied?

*Your answer:* 

**✍️ Question:** Besides ordering materials, name two actions in a CE/ARCE workflow that an agent should never take without human approval, and one guardrail other than approval that would reduce the risk.

*Your answer:* 

In [ ]:
# Free memory at the end of the lab
del llm, tok
gc.collect()

### Looking ahead: the final project
Teams in the **Assist** track build a tool like this one for a real engineering workflow, **with an evaluation**:
a test set with known answers, a baseline (for example, the rule-based brain or a plain function), and an honest
success table like the one in Part 4.

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** Name the four parts of an agent and say in one sentence what each does.

*Your answer:* 

**2.** In your success table, which brain passed more tasks, and give one reason the LLM agent failed (or could fail) a task even though every tool worked correctly.

*Your answer:* 

**3.** What is the difference between the step limit and human approval? Which risk does each one control?

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all.** Allow about 15–20 minutes on a CPU runtime (the model loads again and Part 4 reruns). When the run reaches the **approval box in Part 5, type `no` and press Enter**; otherwise the run waits there.
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.